# Building AI Agents with Supabase and Strands 🔷 🧠

Welcome to this tutorial on integrating Supabase with Strands AI agents! In this notebook, you'll learn how to leverage Supabase's powerful backend services through the Model Context Protocol (MCP) to build intelligent agents that can interact with your database, storage, and edge functions.

## What is Supabase?

[Supabase](https://supabase.com/) is an open-source Firebase alternative that provides all the backend services you need to build a product:

- PostgreSQL database
- Authentication
- Instant APIs
- Edge Functions
- Realtime subscriptions
- Storage

## What is Supabase MCP?

Supabase MCP (Model Context Protocol) is Supabase's implementation of a standardized interface that allows AI models to interact directly with Supabase services. It enables Strands AI agents to seamlessly access and manipulate your Supabase database, storage, authentication, and edge functions through natural language instructions. With Supabase MCP, you can build AI agents that can query your PostgreSQL database, manage files in storage, and trigger serverless functions without writing complex integration code for each specific operation.

## What is Strands?

The AWS Strands Agent Framework enables rapid development of AI agents with minimal code. Strands facilitates building highly dynamic agents through natural language, leveraging prompt engineering to generate varied output types and accept diverse natural language inputs seamlessly.

By the end of this tutorial, you'll know how to:
- Connect Strands agents to Supabase using MCP
- Perform database operations with natural language
- Manage edge functions through AI agents
- Handle storage operations with AI assistance

Let's get started!

## Getting Started

Follow these steps to set up:

1. **Sign up** for Supabase at [supabase.com](https://supabase.com/) and create a project.

2. **Generate a personal access token** from your [Supabase account settings](https://supabase.com/dashboard/account/tokens).

3. **Paste your token** into the cell below and execute the cell.

In [ ]:
# To export your personal access token into a .env file, run the following cell (PLEASE REPLACE WITH YOUR TOKEN):
!echo "SUPABASE_ACCESS_TOKEN=your-personal-access-token" >> .env

Install and import necessary dependencies.

In [ ]:
%pip install strands-agents dotenv --quiet

### Setting Up Your Supabase MCP Client

The code below will instantiate the Supabase MCP client with your personal access token.

In [ ]:
import os
import getpass
from dotenv import load_dotenv
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp import StdioServerParameters, stdio_client

# Load environment variables from .env file
load_dotenv()

# Prompt the user to securely input the personal access token if not already set in the environment
if not os.environ.get("SUPABASE_ACCESS_TOKEN"):
    os.environ["SUPABASE_ACCESS_TOKEN"] = getpass.getpass("SUPABASE_ACCESS_TOKEN:\n")

# Initialize the Supabase MCP client
supabase_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="npx", 
            args=[
                "-y",
                "@supabase/mcp-server-supabase@latest",
                "--access-token", os.getenv("SUPABASE_ACCESS_TOKEN")
            ]
        )
    )
)

## Supabase MCP Architecture

The Supabase MCP integration provides a set of tools that allow AI agents to interact with Supabase services. Here's how it works:

```
┌─────────────────┐      ┌─────────────────┐      ┌─────────────────┐
│                 │      │                 │      │                 │
│   User Input    │──────▶  Strands Agent  │──────▶  MCP Protocol   │
│                 │      │                 │      │                 │
└─────────────────┘      └─────────────────┘      └────────┬────────┘
                                                          │
                                                          │
                                                          ▼
┌─────────────────┐      ┌─────────────────┐      ┌─────────────────┐
│                 │      │                 │      │                 │
│  Edge Functions │◀─────│  Supabase API   │◀─────│  Supabase MCP   │
│                 │      │                 │      │                 │
└─────────────────┘      └─────────────────┘      └─────────────────┘
        ▲                        ▲
        │                        │
        │                        │
┌───────┴────────┐      ┌───────┴────────┐
│                │      │                │
│    Storage     │      │    Database    │
│                │      │                │
└────────────────┘      └────────────────┘
```

The flow works as follows:

1. The Strands agent receives a natural language request from the user
2. The agent determines which Supabase service to use based on the request
3. The agent calls the appropriate MCP tool with the necessary parameters
4. The Supabase MCP server executes the request against the Supabase API
5. The Supabase API interacts with the requested service (Database, Storage, or Edge Functions)
6. The results are returned to the agent, which formats them for the user

This architecture allows for seamless integration between AI agents and Supabase services, enabling natural language interactions with your database, storage, and edge functions.

## Creating a Strands Agent with Supabase MCP Tools

Now, let's create a Strands agent that can use the Supabase MCP tools. We'll use the AWS Bedrock Claude model for this example, but you can use any model supported by Strands.

In [ ]:
from strands.models import BedrockModel

# Initialize the Bedrock model
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name="us-west-2",
)

# Create a system prompt for the agent
system_prompt = """
You are a Supabase expert assistant that helps users manage their Supabase projects, databases, edge functions, and storage.
You have access to Supabase MCP tools that allow you to interact with Supabase services.

When helping users:
1. Always explain what you're doing and why
2. Provide code examples when relevant
3. Suggest best practices for Supabase usage
4. Be security-conscious and avoid exposing sensitive data

You can help with:
- Database operations (SQL queries, table management)
- Edge function deployment and management
- Storage operations (file upload/download, bucket management)
- Project settings and configuration
"""

# Create the agent with Supabase MCP tools
with supabase_client:
    # Get the available tools from the Supabase MCP server
    tools = supabase_client.list_tools_sync()
    
    # Create the agent with the tools and system prompt
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=tools
    )

## Exploring Available Supabase MCP Tools

Let's explore the tools available through the Supabase MCP server. This will help us understand what capabilities our agent has.

In [ ]:
# List all available tools
with supabase_client:
    tools = supabase_client.list_tools_sync()
    
    # Print tool names and descriptions
    for tool in tools:
        print(f"Tool: {tool.name}")
        print(f"Description: {tool.description}")
        print("Parameters:")
        for param_name, param in tool.parameters.get("properties", {}).items():
            print(f"  - {param_name}: {param.get('description', 'No description')}")
        print("\n" + "-"*50 + "\n")

## Database Management Examples

Let's start by exploring how our agent can help with database management tasks. We'll ask it to perform various operations on our Supabase database.

In [ ]:
# Example 1: List all tables in the database
with supabase_client:
    response = agent("List all tables in my Supabase database and explain what each one is used for.")
    print(response)

In [ ]:
# Example 2: Create a new table
with supabase_client:
    response = agent("""
    Create a new table called 'products' with the following columns:
    - id (integer, primary key)
    - name (text, not null)
    - description (text)
    - price (decimal, not null)
    - created_at (timestamp with time zone, default to current timestamp)
    """)
    print(response)

In [ ]:
# Example 3: Insert data into the table
with supabase_client:
    response = agent("""
    Insert the following products into the 'products' table:
    1. Name: 'Laptop', Description: 'High-performance laptop', Price: 999.99
    2. Name: 'Smartphone', Description: 'Latest model with great camera', Price: 699.99
    3. Name: 'Headphones', Description: 'Noise-cancelling wireless headphones', Price: 199.99
    """)
    print(response)

In [ ]:
# Example 4: Query data with filters
with supabase_client:
    response = agent("Find all products with a price greater than 500 dollars and show their names and prices.")
    print(response)

## Edge Function Management Examples

Now, let's explore how our agent can help with edge function management. Edge functions are serverless functions that run on Supabase's edge network.

In [ ]:
# Example 1: List all edge functions
with supabase_client:
    response = agent("List all edge functions in my Supabase project.")
    print(response)

In [ ]:
# Example 2: Create a new edge function
with supabase_client:
    response = agent("""
    Create a new edge function called 'hello-world' that returns a JSON response with the following structure:
    {
        "message": "Hello, World!",
        "timestamp": <current timestamp>
    }
    """)
    print(response)

In [ ]:
# Example 3: Invoke an edge function
with supabase_client:
    response = agent("Invoke the 'hello-world' edge function and show me the response.")
    print(response)

## Storage Management Examples

Finally, let's explore how our agent can help with storage management. Supabase Storage allows you to store and serve files like images, videos, and documents.

In [ ]:
# Example 1: List all storage buckets
with supabase_client:
    response = agent("List all storage buckets in my Supabase project.")
    print(response)

In [ ]:
# Example 2: Create a new storage bucket
with supabase_client:
    response = agent("Create a new public storage bucket called 'images'.")
    print(response)

In [ ]:
# Example 3: Upload a file to storage
# First, let's create a sample file
with open("sample.txt", "w") as f:
    f.write("This is a sample file for Supabase Storage.")

with supabase_client:
    response = agent("Upload the file 'sample.txt' to the 'images' bucket.")
    print(response)

In [ ]:
# Example 4: List files in a bucket
with supabase_client:
    response = agent("List all files in the 'images' bucket.")
    print(response)

## Building a Complete Application

Now that we've explored the individual capabilities of our Supabase-powered agent, let's build a more complete application. We'll create a simple product catalog system with the following features:

1. Database tables for products and categories
2. Edge functions for product recommendations
3. Storage for product images

Let's ask our agent to help us build this application step by step.

In [ ]:
# Build a complete product catalog application
with supabase_client:
    response = agent("""
    I want to build a product catalog application with Supabase. Please help me with the following:
    
    1. Create a 'categories' table with id, name, and description columns
    2. Create a 'products' table with id, name, description, price, category_id (foreign key to categories), and image_url columns
    3. Insert some sample categories and products
    4. Create a storage bucket for product images
    5. Create an edge function that returns products by category
    
    Please explain each step and provide the necessary SQL queries and code.
    """)
    print(response)

## Conclusion

In this tutorial, we've explored how to integrate Supabase with Strands AI agents using the Model Context Protocol (MCP). We've seen how to:

1. Set up a Supabase MCP client
2. Create a Strands agent with Supabase tools
3. Perform database operations with natural language
4. Manage edge functions through AI agents
5. Handle storage operations with AI assistance
6. Build a complete application using all these capabilities

This integration opens up exciting possibilities for building AI-powered applications with Supabase as the backend. You can now create intelligent agents that can interact with your database, storage, and edge functions using natural language, making development faster and more intuitive.

## Next Steps

Here are some ideas for further exploration:

1. Integrate authentication and user management
2. Build a chatbot that can query your Supabase database
3. Create an AI assistant for database schema design
4. Develop a content management system with AI-powered content generation
5. Implement real-time features using Supabase's realtime subscriptions

Happy building with Supabase and Strands! 🔷 🧠